In [7]:
import pandas as pd
from pathlib import Path

# Folder with the raw files (".." means one level up from notebooks)
RAW = Path("../data/raw")

# Find the two files by name, even inside subfolders
artworks_path = next(RAW.rglob("artworks.csv"))
artists_path = next(RAW.rglob("artists.csv"))

# Read both files into tables
artworks = pd.read_csv(artworks_path)
artists = pd.read_csv(artists_path)

print("Artworks:", artworks.shape)
print("Artists:", artists.shape)

Artworks: (130262, 21)
Artists: (15091, 6)


In [8]:

artists_clean = artists.copy() 

artists_clean["Gender"] = artists_clean["Gender"].str.strip().str.capitalize()
artists_clean["Nationality"] = artists_clean["Nationality"].str.strip()


artists_clean["Gender"] = artists_clean["Gender"].fillna("Unknown")
artists_clean["Nationality"] = artists_clean["Nationality"].fillna("Unknown")

print(artists_clean["Gender"].value_counts())

Gender
Male       9826
Unknown    3072
Female     2193
Name: count, dtype: int64


In [9]:

name = artists_clean["Name"].fillna("")
unknown_gender = artists_clean["Gender"] == "Unknown"

anonymous = unknown_gender & name.str.contains(
    "unknown|various|album-miscellaneous", case=False)

group_words = (r"\b(inc|committee|department|institute|collection|corps|brigade|"
               r"center|centre|association|studio|studios|group|company|press|"
               r"agency|society|nasa|photos|workshop|school|survival)\b")
group_names = ["Destroy All Monsters", "Residents", "Proletkult, Leningrad"]
group = unknown_gender & (name.str.contains(group_words, case=False)
                          | name.isin(group_names))

artists_clean["author_type"] = "Individual"
artists_clean.loc[group, "author_type"] = "Group or organization"
artists_clean.loc[anonymous, "author_type"] = "Anonymous"

print(artists_clean.loc[unknown_gender, "author_type"].value_counts())

author_type
Individual               2667
Group or organization     339
Anonymous                  66
Name: count, dtype: int64


C:\Users\22209\AppData\Local\Temp\ipykernel_18412\1780215025.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  group = unknown_gender & (name.str.contains(group_words, case=False)


In [10]:

bridge = artworks[["Artwork ID", "Artist ID"]].copy()
bridge = bridge.dropna(subset=["Artist ID"])

bridge["Artist ID"] = bridge["Artist ID"].astype(str).str.split(",")
bridge = bridge.explode("Artist ID")
bridge["Artist ID"] = bridge["Artist ID"].str.strip()

artists_clean["Artist ID"] = artists_clean["Artist ID"].astype(str).str.strip()

print("Pairs:", len(bridge))
print("Unique artworks:", bridge["Artwork ID"].nunique())
print("Unique artists:", bridge["Artist ID"].nunique())
print("Artists not found in artists table:",
      (~bridge["Artist ID"].isin(artists_clean["Artist ID"])).sum())

Pairs: 142281
Unique artworks: 128802
Unique artists: 13958
Artists not found in artists table: 0


In [11]:

bridge_full = bridge.merge(
    artists_clean[["Artist ID", "Name", "Nationality", "Gender",
                   "Birth Year", "author_type"]],
    on="Artist ID", how="left")

print(bridge_full["Gender"].value_counts(normalize=True).mul(100).round(1))

Gender
Male       79.8
Female     13.6
Unknown     6.6
Name: proportion, dtype: float64


In [12]:

art = artworks[["Artwork ID", "Title", "Date", "Medium", "Classification",
                "Department", "Credit", "Acquisition Date",
                "Height (cm)", "Width (cm)"]].copy()

art["acq_date"] = pd.to_datetime(art["Acquisition Date"], format="mixed", errors="coerce")
art["acq_year"] = art["acq_date"].dt.year

art["create_year"] = art["Date"].astype(str).str.extract(r"(\d{4})")[0].astype(float)

art["date_approx"] = art["Date"].astype(str).str.contains(
    r"^c\.|\bca\.|circa|\?", case=False, regex=True)

print("Acquisition year found:", art["acq_year"].notna().sum())
print("Creation year found:", art["create_year"].notna().sum())

Acquisition year found: 124798
Creation year found: 126811


In [13]:

for col in ["Height (cm)", "Width (cm)"]:
    art[col] = art[col].where(art[col] > 0)

art["area_cm2"] = art["Height (cm)"] * art["Width (cm)"]        # size of the artwork
art["years_to_acquire"] = art["acq_year"] - art["create_year"]   # years until the museum bought it
art["acq_before_create"] = art["years_to_acquire"] < 0           # flag for strange cases
art["acq_decade"] = art["acq_year"] // 10 * 10                   # 1985 -> 1980
art["create_decade"] = art["create_year"] // 10 * 10

print("Height known:", art["Height (cm)"].notna().sum())
print("Width known:", art["Width (cm)"].notna().sum())
print("Median years to acquire:", art["years_to_acquire"].median())
print("Bought before created:", art["acq_before_create"].sum())

Height known: 109330
Width known: 108584
Median years to acquire: 19.0
Bought before created: 92


In [14]:
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

artworks_out = art.rename(columns={
    "Artwork ID": "artwork_id", "Title": "title", "Date": "date_text",
    "Medium": "medium", "Classification": "classification",
    "Department": "department", "Credit": "credit",
    "Height (cm)": "height_cm", "Width (cm)": "width_cm"})

artworks_cols = ["artwork_id", "title", "date_text", "create_year", "date_approx",
                 "acq_date", "acq_year", "acq_decade", "create_decade",
                 "years_to_acquire", "acq_before_create", "classification",
                 "department", "medium", "credit", "height_cm", "width_cm", "area_cm2"]
artworks_out = artworks_out[artworks_cols].copy()

for c in ["create_year", "acq_year", "acq_decade", "create_decade", "years_to_acquire"]:
    artworks_out[c] = artworks_out[c].astype("Int64")

artists_out = artists_clean.rename(columns={
    "Artist ID": "artist_id", "Name": "name", "Nationality": "nationality",
    "Gender": "gender", "Birth Year": "birth_year", "Death Year": "death_year"})
artists_out = artists_out[["artist_id", "name", "nationality", "gender",
                           "birth_year", "death_year", "author_type"]].copy()
for c in ["birth_year", "death_year"]:
    artists_out[c] = artists_out[c].astype("Int64")

bridge_out = bridge.rename(columns={"Artwork ID": "artwork_id", "Artist ID": "artist_id"})
bridge_out = bridge_out.drop_duplicates()

artworks_out.to_csv(PROCESSED / "artworks_clean.csv", index=False)
artists_out.to_csv(PROCESSED / "artists_clean.csv", index=False)
bridge_out.to_csv(PROCESSED / "artwork_artists.csv", index=False)

print("Saved to:", PROCESSED.resolve())
print("artworks:", artworks_out.shape)
print("artists:", artists_out.shape)
print("bridge:", bridge_out.shape)

Saved to: C:\Users\22209\moma-capstone\data\processed
artworks: (130262, 18)
artists: (15091, 7)
bridge: (142176, 2)


In [15]:
from pathlib import Path

p = Path("../data/processed").resolve()
print(p)
print([f.name for f in p.iterdir()])

C:\Users\22209\moma-capstone\data\processed
['.ipynb_checkpoints', 'artists_clean.csv', 'artworks_clean.csv', 'artwork_artists.csv', 'old_artworks_selected.csv']


In [1]:
print("Pairs before removing duplicates:", len(bridge))
print("Duplicate pairs:", bridge.duplicated().sum())
print("Pairs after:", len(bridge.drop_duplicates()))

NameError: name 'bridge' is not defined

In [2]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
artworks = pd.read_csv(next(RAW.rglob("artworks.csv")))

pairs = artworks[["Artwork ID", "Artist ID"]].dropna(subset=["Artist ID"]).copy()
pairs["Artist ID"] = pairs["Artist ID"].astype(str).str.split(",")
pairs = pairs.explode("Artist ID")
pairs["Artist ID"] = pairs["Artist ID"].str.strip()

print("Pairs before removing duplicates:", len(pairs))
print("Duplicate pairs:", pairs.duplicated().sum())
print("Pairs after:", len(pairs.drop_duplicates()))

Pairs before removing duplicates: 142281
Duplicate pairs: 105
Pairs after: 142176
